# LangChain: Models, Prompts and Output Parsers


## Outline

 * Direct API calls to OpenAI
 * API calls through LangChain:
   * Prompts
   * Models
   * Output parsers

In [69]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

_ = load_dotenv(find_dotenv()) # read your local .env file

# --- 1. Set your model (Choose a fast one!) ---
llm_model = "meta/llama-3.1-8b-instruct"

# --- 2. Create the LLM instance ONCE ---
# Set temperature=0 for consistent, factual outputs ideal for learning.
# Set max_tokens to limit response length and save time.
llm = ChatNVIDIA(model=llm_model, 
                 api_key=os.environ['NVIDIA_API_KEY'], 
                 temperature=0,
                 max_tokens=256)


# --- 3. Define a fast, simple prompt function ---
def get_completion(prompt):
    return llm.invoke(prompt).content

/tmp/ipykernel_7116/3006961159.py:13: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model=llm_model,


In [70]:
get_completion("What is 1+1?")

'1 + 1 = 2.'

In [71]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [72]:
style = """American English \
in a calm and respectful tone
"""

In [73]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [74]:
response = get_completion(prompt)

In [75]:
response

'Here\'s the translation:\n\n"I\'m extremely frustrated that my blender lid flew off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn\'t cover the cost of cleaning up my kitchen. I really need your help right now."'

In [76]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

In [77]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatNVIDIA(temperature=0.0, model=llm_model, api_key=os.environ['NVIDIA_API_KEY'])
chat

ChatNVIDIA(output_version=None, profile={}, base_url='https://integrate.api.nvidia.com/v1', model='meta/llama-3.1-8b-instruct', temperature=0.0, default_headers={}, model_kwargs={})

### Prompt template

In [78]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [79]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)


In [80]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [81]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [82]:
customer_style = """American English \
in a calm and respectful tone
"""

In [83]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [84]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [85]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [86]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [87]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)

In [88]:
print(customer_response.content)

Here's the translation of the text into American English in a calm and respectful tone:

"I'm really frustrated that my blender lid flew off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I could really use your help right now."


In [89]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [90]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [91]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [92]:
service_response = chat.invoke(service_messages)
print(service_response.content)

`Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!`

Arrr, me hearty customer, I be havin' some bad news fer ye. Yer warranty don't cover the cost o' cleanin' up the mess ye made in yer kitchen. It seems ye had a wee bit o' trouble with yer blender, matey. It appears ye forgot to put the lid on before startin' it up, savvy? That be considered misuse, me hearty, and we can't be helpin' ye out with the cleanin' bill. Sorry to be tellin' ye this, but it be the way o' the warranty, matey.


## Output Parsers

Let's start with defining how we would like the LLM output to look like:

In [93]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [94]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [95]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)


input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [96]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatNVIDIA(temperature=0.0, model=llm_model, api_key=os.environ['NVIDIA_API_KEY'])
response = chat.invoke(messages)
print(response.content)

```json
{
  "gift": True,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}
```

Explanation:

- The item was purchased as a gift for someone else, so `gift` is `True`.
- The product arrived in 2 days, so `delivery_days` is 2.
- The price or value information is found in the sentence "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.", so `price_value` is a list containing this sentence.


In [97]:
type(response.content)

str

### Parse the LLM output string into a Python dictionary

In [114]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

In [115]:
class ProductReview(BaseModel):
    gift: bool = Field(description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.")
    delivery_days: int = Field(description="How many days did it take for the product to arrive? If this information is not found, output -1.")
    price_value: List[str] = Field(description="Extract any sentences about the value or price, and output them as a list.")

output_parser = PydanticOutputParser(pydantic_object=ProductReview)

In [116]:
format_instructions = output_parser.get_format_instructions()

In [117]:
format_instructions = output_parser.get_format_instructions()

In [118]:
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"gift": {"description": "Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.", "title": "Gift", "type": "boolean"}, "delivery_days": {"description": "How many days did it take for the product to arrive? If this information is not found, output -1.", "title": "Delivery Days", "type": "integer"}, "price_value": {"description": "Extract any sentences about the value or price, and output them as a list.", "items": {"type": "string"}, "title": "Price Value", "type": "array"}}, "required": 

In [119]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}

Output ONLY valid JSON, no additional text or explanation."""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

In [120]:
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties

In [121]:
response = chat.invoke(messages)

In [122]:
print(response.content)

{
  "gift": false,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}


In [123]:
import json
import re

# Extract JSON from response that may contain markdown blocks and explanations
response_text = response.content

# Try to extract JSON block first
json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', response_text)
if json_match:
    json_str = json_match.group(1).strip()
else:
    # If no markdown block, try to extract raw JSON
    json_match = re.search(r'\{[\s\S]*\}', response_text)
    if json_match:
        json_str = json_match.group(0).strip()
    else:
        json_str = response_text

# Fix Python boolean literals to JSON-compatible format
json_str = re.sub(r'\bTrue\b', 'true', json_str)
json_str = re.sub(r'\bFalse\b', 'false', json_str)
json_str = re.sub(r'\bNone\b', 'null', json_str)

# Parse as JSON dictionary
output_dict = json.loads(json_str)

In [124]:
output_dict

{'gift': False,
 'delivery_days': 2,
 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}

In [125]:
type(output_dict)

dict

In [126]:
output_dict.get('delivery_days')

2